# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Access the metadata as an object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` fields.

In [ ]:
# List all record sets by their @id
print("Available record sets (by @id):")
for record_set in dataset.record_sets:
    print(f"- {record_set['@id']}: {record_set.get('name')}")

# For each record set, list the field @id and names
for record_set in dataset.record_sets:
    print(f"\nRecordSet: {record_set['@id']} ({record_set.get('name')})")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, str):
            field_obj = dataset.field_by_id(field)
            print(f"  Field @id: {field_obj['@id']}, name: {field_obj.get('name')}, dataType: {field_obj.get('dataType')}")
        elif isinstance(field, dict):
            print(f"  Field @id: {field.get('@id')}, name: {field.get('name')}, dataType: {field.get('dataType')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Pick all available record set IDs for extraction
record_set_ids = [r['@id'] for r in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for {record_set_id}")
    else:
        print(f"No records found for {record_set_id}.\n")

# For demonstration, select the first available record set with data
main_record_set_id = next(iter(dataframes)) if dataframes else None
if main_record_set_id:
    print(f"\nColumns in main record set ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print("No record sets with data found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

You can replace the `<numeric_field_id>` and `<group_field_id>` values with relevant field `@id`s from your record set.

In [ ]:
# Identify numeric and group-able fields for the main record set
import numpy as np

if main_record_set_id:
    df = dataframes[main_record_set_id]

    # Attempt auto-detection of numeric fields
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric fields in {main_record_set_id}: {numeric_fields}")

    # Use the first numeric field for demonstration, or specify by @id
    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # e.g., 'cr:coverage' or actual column name
        threshold = df[numeric_field_id].quantile(0.75)  # Use 75th percentile as threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold} (75th percentile):")
        print(filtered_df.head())

        # Normalize the numeric field
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, normalized_col]].head())

        # Attempt grouping by the first non-numeric field (if available)
        group_fields = [c for c in df.columns if c not in numeric_fields]
        group_field_id = group_fields[0] if group_fields else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric fields available for filtering and normalization.")
else:
    print("No main record set available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_fields:
    # Distribution plot for the main numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in {main_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If a group field exists, boxplot of numeric field by group
    if group_field_id:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No sufficient data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 Croissant dataset for mass spectrometry analysis of extracellular vesicles in human mast cells was successfully loaded and explored.
- Data fields, such as protein coverage or peptide counts, can be filtered and normalized for downstream analysis.
- Plots illustrate the distributions and group differences in key numeric features.

You can further extend this analysis by selecting additional record sets, fields, and applying machine learning models as appropriate.